# 06 - Sensor design for Belauri: where should a few sensors go?

**Real geography; synthetic emissions and meteorology.** The municipality polygon, the road / built-up / facility geometry and the Nepal outline are real (prepared in `data/belauri`). Every emission rate, the inflow, every wind and all noise are **synthetic**. Candidate sites, layouts and every error below are a **provisional simulation design, not field-validated**: mapped accessibility is not permission, power, security or connectivity, and gaps in OpenStreetMap are not proof that a site is infeasible.

**What runs here (offline, bounded).** The *smoke* profile of the study (`configs/geodesign_smoke.json`: outer domain cropped to a 2 km buffer, T = 7200 s, 2 + 2 + 2 cases, a coarse 3 km candidate lattice, budgets 1 and 3) is executed in this notebook, followed by the smoke profile of the robustness study (`configs/geodesign_robust_smoke.json`). It checks the plumbing and illustrates the outputs; **it is too small for conclusions**. If a production run is present under `results/geodesign` (and `results/geodesign-robust`), its saved tables and figures are shown as well (optional cells).

Terms. *Candidate*: a possible sensor position. *Layout*: a set of candidates (budget = its size). *Greedy*: add one site at a time, always the one that most lowers the design objective: a heuristic, never called optimal. *Total error*: study-area x time weighted RMSE of the reconstructed concentration. *Attribution error*: the same for the local and external parts of the field, each divided by the mean square of that true component (design cases only), then averaged.

In [ ]:
import json, tempfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from pravah import geodesign as G
from pravah import geodesign_robust as R
from pravah.belauri import draw_map
from pravah.config import load_config
from pravah.inverse import ResponseCache
%matplotlib inline
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
T0 = time.perf_counter()
root = Path.cwd()
while not (root / "configs" / "geodesign_smoke.json").exists() and root != root.parent:
    root = root.parent
LABEL = "Real geography; synthetic emissions and meteorology."
work = Path(tempfile.mkdtemp(prefix="pravah_nb06_"))
cache = ResponseCache()
cfg = load_config(root / "configs" / "geodesign_smoke.json")
print("work dir (temporary):", work.name)

## 1. Run the smoke design study

Order of the pipeline: candidates, then design and validation ensembles, then alpha (ridge strength) chosen on the **validation** ensemble only, then layouts chosen on the **design** ensemble only, and only then the independent **test** ensemble with fresh noise. Test seeds never influence layouts or alpha.

In [ ]:
t0 = time.perf_counter()
main_dir = work / "geodesign"
G.run_geodesign(cfg, main_dir, cache=cache)
print(f"smoke geodesign run: {time.perf_counter() - t0:.0f} s")
st = G.Setup(cfg, cache)
fixed = R.load_fixed_design(main_dir, st)
summ = pd.read_csv(main_dir / "summary.csv")
paired = pd.read_csv(main_dir / "paired.csv")
layouts = pd.read_csv(main_dir / "layouts.csv")
print("files:", sorted(p.name for p in main_dir.iterdir()))

## 2. Candidate sites

Interior sites are a regular lattice clipped to the municipality polygon; **reference** sites lie outside it but inside Nepal and the simulation domain (at most 2 may be used, counted in the budget). Sites across the national border are *hypothetical* and **excluded by default**.

In [ ]:
tab = st.table
fig, ax = plt.subplots(figsize=(6.2, 5.6))
draw_map(ax, st.geo.region, st.res)
cat = np.asarray(st.cands.categories)
for name, col, mk in (("interior", "tab:blue", "o"), ("reference_domestic", "tab:orange", "s")):
    m = cat == name
    ax.scatter(st.cands.x[m] / 1000, st.cands.y[m] / 1000, s=45, c=col, marker=mk, edgecolors="k", label=f"{name} ({m.sum()})", zorder=4)
ax.set_title("Candidate pool used by the experiments", fontsize=9); ax.legend(fontsize=8)
fig.text(0.01, 0.005, LABEL + " Provisional simulation design.", fontsize=7, style="italic")
plt.show()
print(tab["category"].value_counts().to_dict(), "| hypothetical (excluded by default):", int(tab["hypothetical"].sum()))
print(tab[["id", "category", "in_polygon", "dist_road_m", "dist_facility_m", "on_or_near_road"]].head(6).round(0))
print("assumptions:")
for a in G.ASSUMPTIONS:
    print("  -", a)

## 3. Selected sites: spread, greedy_total, greedy_attribution

The same budget and noise fraction; spread is the farthest-point rule (no simulation), the two greedy layouts use the design cases with different objectives.

In [ ]:
f0 = st.fractions[0]
b = max(st.budgets)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.9), layout="constrained")
for ax, meth in zip(axes, ("spread", "greedy_total", "greedy_attribution")):
    draw_map(ax, st.geo.region, st.res)
    ax.scatter(st.cands.x / 1000, st.cands.y / 1000, s=6, c="0.85", edgecolors="0.4", linewidths=0.3, zorder=3)
    rec = next(r for r in fixed.layouts if (r.setting, r.regime, r.method, r.budget, r.k) == ("interior_only", "standard", meth, b, 0)
               and r.noise_fraction in (None, f0))
    ax.scatter(st.cands.x[list(rec.layout)] / 1000, st.cands.y[list(rec.layout)] / 1000, s=90, c="tab:blue", edgecolors="k", zorder=5)
    for pos, i in enumerate(rec.order or rec.layout):
        ax.annotate(str(pos + 1) if rec.order else "", (st.cands.x[i] / 1000, st.cands.y[i] / 1000), color="w", fontsize=7, ha="center", va="center", zorder=6)
    ax.set_title(f"{meth} (budget {b}); numbers = order added" if rec.order else f"{meth} (budget {b})", fontsize=9)
fig.text(0.01, 0.005, LABEL + " Provisional simulation design; greedy layouts are heuristics.", fontsize=7, style="italic")
plt.show()

## 4. Error versus number of sensors

Total regional RMSE on the **independent test ensemble** (fresh noise), mean over layouts x cases x noise repetitions; the shaded band is **+/- 1 s.d.** over those rows. It is descriptive: cases and repetitions are not independent draws, so it is never a confidence interval. (Smoke: 2 cases, 2 repetitions.)

In [ ]:
fig = G.fig_error_curves(summ, "interior_only", "standard"); plt.show()
view = summ[(summ.setting == "interior_only") & (summ.regime == "standard") & (summ.alpha_label == "selected")]
display(view.groupby(["noise_fraction", "method", "budget"])[["mean_rmse", "sd_rmse", "mean_rmse_local", "mean_rmse_external", "mean_attr_err"]].mean().round(4))

## 5. Does optimising one objective hurt the other?

`greedy_attribution` minimises the normalised local + external error; `greedy_total` the total field error. Both are scored on **both** metrics. Negative paired differences mean the first method is better; `a_wins_cases` counts the test cases won (a handful in the smoke run).

In [ ]:
fig = G.fig_attribution_vs_total(summ, "interior_only", "standard"); plt.show()
p = paired[(paired.comparison == "greedy_attribution vs greedy_total") & (paired.setting == "interior_only") & (paired.regime == "standard")]
display(p[["noise_fraction", "budget", "metric", "mean_diff", "sd_diff", "n_pairs", "a_wins_cases", "n_cases"]].round(4))

## 6. Value of external reference sites at equal budget

`with_reference` may place up to two sensors outside the polygon (counted in the budget); the comparison is paired with `interior_only` on the same cases and noise.

In [ ]:
p = paired[(paired.comparison == "with_reference vs interior_only") & (paired.regime == "standard") & (paired.metric == "rmse")]
display(p[["method_a", "noise_fraction", "budget", "mean_diff", "sd_diff", "n_pairs", "a_wins_cases", "n_cases"]].round(4))
sel = pd.read_csv(main_dir / "test_raw.csv")
sel = sel[(sel.alpha_label == "selected") & (sel.setting == "with_reference") & (sel.regime == "standard")]
print("mean number of reference sites per layout, by method:", sel.groupby("method")["n_reference_sites"].mean().round(2).to_dict())

## 7. Spatially separated validation

A block of the study area (the east third) is held out. In the `block_holdout` regime every layout keeps an **exclusion radius** (2000 m) from the block's evaluation points, so the block error is a true hold-out; budgets that become impossible are reported, not dropped.

In [ ]:
fig = G.fig_block_map(st, fixed.layouts, f0); plt.show()
print("infeasible budgets:", json.loads((main_dir / "infeasible_budgets.json").read_text())["infeasible"])
v = summ[(summ.regime == "block_holdout") & (summ.alpha_label == "selected") & (summ.noise_fraction == f0)]
display(v.groupby(["setting", "method", "budget"])[["mean_heldout_block_rmse", "mean_heldout_random_rmse", "mean_rmse"]].mean().round(4))

## 8. Robustness of the fixed layouts (smoke version of `geodesign_robust`)

The layouts and the frozen alpha above are **re-used verbatim** (nothing is re-selected) on the regenerated test ensemble (same seeds and noise, so the matched rows are exactly the rows above) under separate conditions: unseen winds (declared as nominal / declared correctly), a hidden wind error, a different boundary time profile, an omitted local facility source, an omitted external Gaussian source, and a 2x finer truth block-averaged to the inference grid. Mismatch need not worsen every draw; paired differences and per-case counts are reported.

In [ ]:
t0 = time.perf_counter()
rcfg = load_config(root / "configs" / "geodesign_robust_smoke.json")
rob_dir = work / "robust"
res = R.run_geodesign_robust(rcfg, rob_dir, layouts_from=main_dir, cache=cache)
print(f"smoke robustness run (includes refinement): {time.perf_counter() - t0:.0f} s")
ps = pd.read_csv(rob_dir / "paired_mismatch.csv")
x = ps[(ps.metric == "rmse") & (ps.method == "greedy_total") & (ps.setting == "interior_only") & (ps.noise_fraction == f0)]
display(x[["condition", "budget", "mean_matched", "mean_condition", "mean_diff", "sd_diff", "n_cases_worse", "n_cases"]].round(4))
cases = pd.read_csv(rob_dir / "cases.csv")
display(cases[["case", "condition", "truth_wind", "nominal_wind", "max_courant_truth", "unmodelled_emission_share"]].round(3))
print("matched rows equal the main run's:", json.loads((rob_dir / "provenance.json").read_text())["extra"]["matched_vs_main_test_raw"])

In [ ]:
display(Image(filename=str(rob_dir / "figures" / "rmse_by_condition_interior_only.png")))
display(Image(filename=str(rob_dir / "figures" / "paired_difference.png")))
con = pd.read_csv(rob_dir / "ranking_changes.csv")
c = con[(con.comparison == "greedy_total vs spread") & (con.metric == "rmse") & (con.setting == "interior_only") & (con.noise_fraction == f0)]
print("Does greedy_total still beat spread under each condition? (negative = yes)")
display(c[["condition", "budget", "mean_diff", "mean_diff_matched", "same_sign_as_matched", "a_wins_cases", "n_cases"]].round(4))

### Identifiability flags cannot detect a wrong model

The flags (`rank_deficient`, `weakly_constrained`, `unidentifiable:*`, `confounded:*`) are computed from the operator (the sensor sites and the model's responses) and sigma. They do not look at the data, so under a wrong wind, a different boundary profile or an omitted source they stay exactly as in the matched case while the error moves. The table puts the flagged fraction next to the error ratio.

In [ ]:
fl = pd.read_csv(rob_dir / "flags_vs_error.csv")
display(fl[(fl.method == "greedy_total")][["condition", "budget", "frac_flagged", "frac_flagged_matched", "frac_flags_equal_matched", "mean_rmse", "mean_rmse_matched", "rmse_ratio_to_matched"]].round(3))

## 9. Numerical-refinement sensitivity

Two checks. (i) Layouts chosen on the 500 m problem are evaluated against a 250 m / 50 s truth (the `refined_truth` condition above; sensors read the fine field, the regional error uses its 500 m block average). Because the truth uses the same scheme on a finer mesh, this is a **lower bound** on the inverse-crime effect: the matched experiments share their numerics, and the contrast shows how much of the matched accuracy is that. (ii) Greedy layouts are **re-selected on the 250 m model** (here 1 design case, budgets 1 and 3) and compared with the 500 m layouts (Jaccard index, nearest-site distance) and on the same fine test truth. At production size each 250 m response set is about 8 solves of ~30 s.

In [ ]:
ov = pd.read_csv(rob_dir / "refinement_overlap.csv")
display(ov[["setting", "method", "budget", "noise_fraction", "jaccard", "n_shared_sites", "mean_nearest_site_distance_m", "max_nearest_site_distance_m"]].round(2))
gap = pd.read_csv(rob_dir / "refinement_gap.csv")
display(gap[gap.metric == "rmse"][["setting", "noise_fraction", "method", "budget", "mean_500m", "mean_250m", "mean_diff_250m_minus_500m", "cases_won_by_250m_layouts", "n_cases"]].round(4))
display(Image(filename=str(rob_dir / "figures" / "refinement_sites.png")))
nd = pd.read_csv(rob_dir / "numerical_diffusion.csv")
display(nd[nd.speed_label.str.startswith("reference")][["mesh", "dx_m", "dt_s", "speed_m_s", "peclet", "courant", "K_eff_over_K"]].round(2))
print(json.loads((rob_dir / "numerical_diffusion.json").read_text())["note"])

## 10. Identifiability and prior dependence

alpha is the ridge strength in scaled units, chosen on the validation ensemble only; the test results are also given for alpha = 0 (no prior) and a strong prior (alpha = 1000, shrinks towards the prior mean 0). A large gap between the rows shows how much the prior carries. Identifiability flags are reported per row.

In [ ]:
alpha = json.loads((main_dir / "alpha_selected.json").read_text())
print("selected alpha by noise fraction:", alpha["alpha_by_noise_fraction"], "| uses test ensemble:", alpha["uses_test_ensemble"])
v = summ[(summ.setting == "interior_only") & (summ.regime == "standard") & (summ.method == "greedy_total") & (summ.noise_fraction == f0)]
display(v.pivot_table(index="budget", columns="alpha_label", values="mean_rmse").round(4))
d = summ[(summ.alpha_label == "selected") & (summ.regime == "standard")]
print("share of selected-alpha rows with an identifiability flag, by budget:")
display(d.groupby("budget")["frac_rows_id_flagged"].mean().round(2).to_frame())

## 11. Production results (optional)

Shown only if `results/geodesign` (and `results/geodesign-robust`) exist and are complete; otherwise this cell does nothing.

In [ ]:
prod = root / "results" / "geodesign"
if (prod / "summary.csv").exists() and (prod / "figures").is_dir():
    ps_ = pd.read_csv(prod / "summary.csv")
    print("production study:", prod.name, "|", len(pd.read_csv(prod / "layouts.csv")), "layouts")
    v = ps_[(ps_.setting == "interior_only") & (ps_.regime == "standard") & (ps_.alpha_label == "selected")]
    display(v.pivot_table(index="budget", columns=["noise_fraction", "method"], values="mean_rmse").round(4))
    for fig_name in ("selected_sites.png", "error_vs_count_interior_only.png", "attribution_vs_total.png", "heldout_block_map.png"):
        if (prod / "figures" / fig_name).exists():
            display(Image(filename=str(prod / "figures" / fig_name)))
    pr = root / "results" / "geodesign-robust"
    if (pr / "paired_mismatch.csv").exists():
        pm = pd.read_csv(pr / "paired_mismatch.csv")
        x = pm[(pm.metric == "rmse") & (pm.method == "greedy_total") & (pm.setting == "interior_only") & (pm.noise_fraction == pm.noise_fraction.min())]
        display(x.pivot_table(index="budget", columns="condition", values="mean_diff").round(4))
        for fig_name in ("rmse_by_condition_interior_only.png", "paired_difference.png"):
            if (pr / "figures" / fig_name).exists():
                display(Image(filename=str(pr / "figures" / fig_name)))
else:
    print("no complete production run under results/geodesign: nothing to show (the smoke results above stand in)")

## 12. What these results are, and are not

* **Real geography; synthetic emissions and meteorology.** All emissions, winds, inflow and noise are synthetic; only the geometry is real.
* The layouts are **provisional simulation designs, not field-validated deployment instructions.** Mapped accessibility is not permission, power, security or connectivity; OSM gaps are not absence of feasible sites; reference sites are only as realistic as the Nepal ADM0 outline, and anything across the border is hypothetical.
* Greedy layouts are heuristics, not optimal; they depend on the few design cases drawn.
* The matched experiments are idealised (known wind, K, lam, H, source shapes and onsets). The robustness conditions test some departures one at a time; they do not cover terrain-driven flow, unknown source shapes or instrument bias.
* With a handful of cases and noise repetitions the differences between methods are descriptive, not statistical conclusions.
* The physical K = 100 m2/s is not tuned; the numerical spreading of the scheme (K_eff/K above) is reported separately and affects every model in the study.
* A field campaign should treat these layouts as candidates to be re-examined with measurements.

Total notebook time so far: see the next cell.

In [ ]:
print(f"notebook wall time: {time.perf_counter() - T0:.0f} s")